# Select Sentinel2/ECOSTRESS products

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

In [3]:
from sensorsio.mgrs import get_mgrs_tiles_from_roi

In [4]:
from etdataset.provider import Collection
from etdataset.utils import get_bbox_from_roi
from etdataset.api import select
from etdataset.writer import write_matches, write_results

/home/sarrazine/pyenvs/cesbio/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Inputs

To be modified 

In [5]:
# ROI
s2_tile_id = "28PCA"
data_path = os.environ["DATA_PATH"]
roi_path = os.path.join(data_path, "zones", "Zone_Senegal_Centre.shp")

In [6]:
# Acquisition dates
min_date = "2023-02-01"  # Format YYYY-MM-DD
max_date = "2023-03-31"  # Format YYYY-MM-DD

In [7]:
# Delta time between two acquisitions
delta_time = "3 days"

In [8]:
# Other criteria to filter products
max_cloud_cover = 25
min_roi_overlap = 33
min_product_overlap = 40

In [9]:
# Outpit directory
output_path = "out"
os.makedirs(output_path, exist_ok=True)

### To obtain a list of MGRS tiles that overlap a ROI with the percentage of surface area covered

In [10]:
roi_bbox, roi_crs = get_bbox_from_roi(roi_path)
mgrs_tiles = get_mgrs_tiles_from_roi(roi_bbox, roi_crs)
mgrs_tiles

,Name,overlap_geometry,geometry,overlap_percentage
0,28PBA,"POLYGON ((-16.76482 14.46621, -16.75975 13.800...","POLYGON ((-17.78282 14.45638, -16.76482 14.466...",22.271238
1,28PBB,"POLYGON ((-16.76412 14.37783, -17.10000 14.374...","POLYGON ((-17.79446 15.35961, -16.77221 15.370...",20.551890
2,28PCA,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120620
3,28PCB,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627789
4,28PDA,"POLYGON ((-15.90000 14.47105, -15.90000 13.800...","POLYGON ((-15.92826 14.47100, -14.90943 14.472...",1.789794
5,28PDB,"POLYGON ((-15.92790 14.38258, -15.93054 15.000...","POLYGON ((-15.93215 15.37519, -14.90905 15.377...",1.780280


### Select products on a MGRS tile

In [11]:
df1, df2, matches = select(
    Collection.SENTINEL2,
    Collection.ECOSTRESS,
    min_date,
    max_date,
    tile_id=s2_tile_id,
    delta=delta_time,
    max_cloud_cover=max_cloud_cover,
    min_roi_overlap=min_roi_overlap,
    min_product_overlap=min_product_overlap,
    only_best_match=True
)

INFO:theia-picker:Renew token
INFO:etdataset.api:Products found for SENTINEL2 in catalog: 10
INFO:etdataset.api:Products found for SENTINEL2 after ROI filtering: 10


Granules found: 127


INFO:etdataset.api:Products found for ECOSTRESS in catalog: 12
INFO:etdataset.api:Products found for ECOSTRESS after ROI filtering: 12
Matching datasets ...: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 60.18it/s]
INFO:etdataset.selection:Found 7 matches


In [12]:
matches

,Product_name_SENTINEL2,Date_SENTINEL2,Product_name_ECOSTRESS,Date_ECOSTRESS,overlap
0,SENTINEL2A_20230226-114734-900_L2A_T28PCA_D,2023-02-26,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,98.94908
1,SENTINEL2A_20230308-114732-624_L2A_T28PCA_D,2023-03-08,ECOv002_L2T_LSTE_26494_015_28PCA_20230309T0806...,2023-03-09,98.94908
2,SENTINEL2A_20230318-114735-293_L2A_T28PCA_D,2023-03-18,ECOv002_L2T_LSTE_26616_002_28PCA_20230317T0458...,2023-03-17,98.94908
3,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,ECOv002_L2T_LSTE_26800_017_28PCA_20230329T0015...,2023-03-29,98.94908
4,SENTINEL2B_20230211-114735-878_L2A_T28PCA_D,2023-02-11,ECOv002_L2T_LSTE_26128_016_28PCA_20230213T1738...,2023-02-13,98.94908
5,SENTINEL2B_20230303-114738-815_L2A_T28PCA_D,2023-03-03,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,98.94908
6,SENTINEL2B_20230323-114739-427_L2A_T28PCA_D,2023-03-23,ECOv002_L2T_LSTE_26678_018_28PCA_20230321T0324...,2023-03-21,98.94908


In [13]:
df1

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,e09e27d1cff4e8c103c3f55fde111dab,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0
1,SENTINEL2B_20230323-114739-427_L2A_T28PCA_D,2023-03-23,THEIA,SENTINEL2,28PCA,12,80,https://theia.cnes.fr/atdistrib/resto2/collect...,55ecd31c7f95ac546df54dae7c49c179,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0
2,SENTINEL2A_20230318-114735-293_L2A_T28PCA_D,2023-03-18,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,1ea5fc4e111821916417d332f8e1508a,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0
3,SENTINEL2A_20230308-114732-624_L2A_T28PCA_D,2023-03-08,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,e9946922d2a1c4b9ca2bebc1e5acc91b,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0
4,SENTINEL2B_20230303-114738-815_L2A_T28PCA_D,2023-03-03,THEIA,SENTINEL2,28PCA,12,80,https://theia.cnes.fr/atdistrib/resto2/collect...,18ba0c4e6569efaafadad69198b27145,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0
5,SENTINEL2A_20230226-114734-900_L2A_T28PCA_D,2023-02-26,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,8e686b861ed93c83a92e53006213ace3,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0
6,SENTINEL2B_20230211-114735-878_L2A_T28PCA_D,2023-02-11,THEIA,SENTINEL2,28PCA,10,80,https://theia.cnes.fr/atdistrib/resto2/collect...,dd7fcc74745991be7426078552d36ffd,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",100.0


In [14]:
df2

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,ECOv002_L2T_LSTE_26128_016_28PCA_20230213T1738...,2023-02-13,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
1,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
2,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
3,ECOv002_L2T_LSTE_26494_015_28PCA_20230309T0806...,2023-03-09,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
4,ECOv002_L2T_LSTE_26616_002_28PCA_20230317T0458...,2023-03-17,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
5,ECOv002_L2T_LSTE_26678_018_28PCA_20230321T0324...,2023-03-21,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0
6,ECOv002_L2T_LSTE_26800_017_28PCA_20230329T0015...,2023-03-29,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.83376 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",100.0


### Select products on a ROI

In [15]:
df1, df2, matches = select(
    Collection.SENTINEL2,
    Collection.ECOSTRESS,
    min_date,
    max_date,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    delta=delta_time,
    max_cloud_cover=max_cloud_cover,
    min_roi_overlap=min_roi_overlap,
    min_product_overlap=min_product_overlap,
    only_best_match=False
)

INFO:theia-picker:Renew token
INFO:etdataset.api:Products found for SENTINEL2 in catalog: 34
INFO:etdataset.api:Products found for SENTINEL2 after ROI filtering: 18


Granules found: 81


INFO:etdataset.api:Products found for ECOSTRESS in catalog: 81
INFO:etdataset.api:Products found for ECOSTRESS after ROI filtering: 25
Matching datasets ...: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 45.08it/s]
INFO:etdataset.selection:Found 32 matches


In [16]:
matches

,Product_name_SENTINEL2,Date_SENTINEL2,Product_name_ECOSTRESS,Date_ECOSTRESS,overlap
0,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,ECOv002_L2T_LSTE_26739_017_28PCA_20230325T0150...,2023-03-25,98.949080
1,SENTINEL2A_20230328-114720-999_L2A_T28PCB_D,2023-03-28,ECOv002_L2T_LSTE_26739_017_28PCB_20230325T0150...,2023-03-25,98.874518
2,SENTINEL2A_20230328-114720-999_L2A_T28PCB_D,2023-03-28,ECOv002_L2T_LSTE_26748_001_28PCB_20230325T1509...,2023-03-25,98.874518
3,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,ECOv002_L2T_LSTE_26748_001_28PCA_20230325T1509...,2023-03-25,98.949080
4,SENTINEL2A_20230328-114720-999_L2A_T28PCB_D,2023-03-28,ECOv002_L2T_LSTE_26800_017_28PCB_20230329T0015...,2023-03-29,98.874518
5,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,ECOv002_L2T_LSTE_26800_017_28PCA_20230329T0015...,2023-03-29,98.949080
6,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,ECOv002_L2T_LSTE_26809_001_28PCA_20230329T1333...,2023-03-29,98.949080
7,SENTINEL2A_20230328-114720-999_L2A_T28PCB_D,2023-03-28,ECOv002_L2T_LSTE_26809_001_28PCB_20230329T1333...,2023-03-29,98.874518
8,SENTINEL2B_20230323-114725-218_L2A_T28PCB_D,2023-03-23,ECOv002_L2T_LSTE_26678_018_28PCB_20230321T0324...,2023-03-21,98.874518
9,SENTINEL2B_20230323-114739-427_L2A_T28PCA_D,2023-03-23,ECOv002_L2T_LSTE_26678_018_28PCA_20230321T0324...,2023-03-21,98.949080


In [17]:
df1

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,SENTINEL2A_20230328-114735-205_L2A_T28PCA_D,2023-03-28,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,e09e27d1cff4e8c103c3f55fde111dab,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120630
1,SENTINEL2A_20230328-114720-999_L2A_T28PCB_D,2023-03-28,THEIA,SENTINEL2,28PCB,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,999804996f06d512984d9f9123cbde33,"POLYGON ((-16.85479 14.37713, -16.86013 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627817
2,SENTINEL2B_20230323-114739-427_L2A_T28PCA_D,2023-03-23,THEIA,SENTINEL2,28PCA,12,80,https://theia.cnes.fr/atdistrib/resto2/collect...,55ecd31c7f95ac546df54dae7c49c179,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120630
3,SENTINEL2B_20230323-114725-218_L2A_T28PCB_D,2023-03-23,THEIA,SENTINEL2,28PCB,17,80,https://theia.cnes.fr/atdistrib/resto2/collect...,c18eee25451b131e53ba8447cd351d00,"POLYGON ((-16.85479 14.37713, -16.86013 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627817
4,SENTINEL2A_20230318-114735-293_L2A_T28PCA_D,2023-03-18,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,1ea5fc4e111821916417d332f8e1508a,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120630
5,SENTINEL2A_20230318-114721-093_L2A_T28PCB_D,2023-03-18,THEIA,SENTINEL2,28PCB,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,999c4860f3e18f969297d39924c253c2,"POLYGON ((-16.85479 14.37713, -16.86013 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627817
6,SENTINEL2A_20230308-114732-624_L2A_T28PCA_D,2023-03-08,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,e9946922d2a1c4b9ca2bebc1e5acc91b,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120630
7,SENTINEL2A_20230308-114718-413_L2A_T28PCB_D,2023-03-08,THEIA,SENTINEL2,28PCB,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,b1896965a3781da5e8df95577114608a,"POLYGON ((-16.85479 14.37713, -16.86013 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627817
8,SENTINEL2B_20230303-114738-815_L2A_T28PCA_D,2023-03-03,THEIA,SENTINEL2,28PCA,12,80,https://theia.cnes.fr/atdistrib/resto2/collect...,18ba0c4e6569efaafadad69198b27145,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120630
9,SENTINEL2A_20230226-114734-900_L2A_T28PCA_D,2023-02-26,THEIA,SENTINEL2,28PCA,0,80,https://theia.cnes.fr/atdistrib/resto2/collect...,8e686b861ed93c83a92e53006213ace3,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120630


In [18]:
df2

,Product_name,Date,Provider,Collection,Tile_ID,Cloud_cover,Relative_orbit,URL,Checksum,overlap_geometry,geometry,overlap_percentage
0,ECOv002_L2T_LSTE_26128_016_28PCA_20230213T1738...,2023-02-13,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
1,ECOv002_L2T_LSTE_26372_016_28PCB_20230301T1116...,2023-03-01,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
2,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
3,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
4,ECOv002_L2T_LSTE_26433_014_28PCA_20230305T0940...,2023-03-05,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
5,ECOv002_L2T_LSTE_26433_014_28PCB_20230305T0940...,2023-03-05,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
6,ECOv002_L2T_LSTE_26433_015_28PCB_20230305T0941...,2023-03-05,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
7,ECOv002_L2T_LSTE_26433_015_28PCA_20230305T0941...,2023-03-05,EarthData,ECOSTRESS,28PCA,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.85552 14.47134, -15.90000 14.471...","POLYGON ((-16.85552 13.47358, -16.85552 14.471...",62.922639
8,ECOv002_L2T_LSTE_26494_014_28PCB_20230309T0805...,2023-03-09,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361
9,ECOv002_L2T_LSTE_26494_015_28PCB_20230309T0806...,2023-03-09,EarthData,ECOSTRESS,28PCB,None,None,https://data.lpdaac.earthdatacloud.nasa.gov/lp...,None,"POLYGON ((-16.86329 15.00000, -15.90000 15.000...","POLYGON ((-16.86329 14.37749, -16.86329 15.375...",58.543361


### Write results

In [19]:
write_matches(matches, os.path.join(output_path,"matches.csv"))
write_results(df1, os.path.join(output_path,f"products_SENTINEL2.csv"))
write_results(df2, os.path.join(output_path,f"products_ECOSTRESS.csv"))